# WebSockets — Práctica

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/18_intro_a_apis/code/03_websockets.ipynb)


## 1. Setup

Instalamos `websockets`, la librería estándar de Python para comunicación WebSocket.

In [ ]:
%pip install websockets requests matplotlib

In [ ]:
import asyncio
import websockets
import requests
import time
import json
import matplotlib.pyplot as plt

## 2. Conexión a un echo server

Un **echo server** es un servidor que devuelve exactamente lo que le enviamos.
Es perfecto para entender cómo funciona WebSocket.

A diferencia de HTTP (request-response), WebSocket mantiene una **conexión
persistente y bidireccional** entre cliente y servidor.

In [ ]:
# Conexión básica a un echo server
ECHO_URL = "wss://ws.postman-echo.com/raw"

async def echo_basico():
    async with websockets.connect(ECHO_URL) as ws:
        # Enviamos un mensaje
        mensaje = "Hola desde ITAM"
        await ws.send(mensaje)
        print(f"Enviado:  '{mensaje}'")

        # Recibimos la respuesta (eco)
        respuesta = await ws.recv()
        print(f"Recibido: '{respuesta}'")

        # Verificamos que es un eco exacto
        print(f"Es eco:   {mensaje == respuesta}")

await echo_basico()

In [ ]:
# También podemos enviar JSON
async def echo_json():
    async with websockets.connect(ECHO_URL) as ws:
        datos = {
            "tipo": "saludo",
            "contenido": "Hola mundo",
            "timestamp": time.time()
        }

        await ws.send(json.dumps(datos))
        respuesta = await ws.recv()
        datos_recibidos = json.loads(respuesta)

        print("Datos enviados y recibidos:")
        print(json.dumps(datos_recibidos, indent=2))

await echo_json()

## 3. Medición de latencia

Medimos el **RTT** (Round-Trip Time): el tiempo desde que enviamos un mensaje
hasta que recibimos la respuesta.

In [ ]:
# Medir RTT de un solo mensaje
async def medir_rtt():
    async with websockets.connect(ECHO_URL) as ws:
        inicio = time.time()
        await ws.send("ping")
        respuesta = await ws.recv()
        rtt = (time.time() - inicio) * 1000  # convertir a milisegundos

        print(f"RTT: {rtt:.1f} ms")
        return rtt

rtt = await medir_rtt()

In [ ]:
# Medir RTT múltiples veces para obtener estadísticas
async def medir_rtts(n=10):
    rtts = []
    async with websockets.connect(ECHO_URL) as ws:
        for i in range(n):
            inicio = time.time()
            await ws.send(f"ping-{i}")
            await ws.recv()
            rtt = (time.time() - inicio) * 1000
            rtts.append(rtt)
    return rtts

rtts_ws = await medir_rtts(10)

print(f"RTTs WebSocket ({len(rtts_ws)} mediciones):")
print(f"  Mínimo:  {min(rtts_ws):.1f} ms")
print(f"  Máximo:  {max(rtts_ws):.1f} ms")
print(f"  Promedio: {sum(rtts_ws)/len(rtts_ws):.1f} ms")

## 4. Múltiples mensajes

Una ventaja clave de WebSocket es que la conexión **persiste**.
No hay overhead de establecer una nueva conexión TCP/TLS para cada mensaje.

In [ ]:
# Enviar muchos mensajes en la misma conexión
async def multiples_mensajes():
    async with websockets.connect(ECHO_URL) as ws:
        mensajes = [
            "Primer mensaje",
            "Segundo mensaje",
            "Tercer mensaje",
            json.dumps({"tipo": "datos", "valor": 42}),
            "Mensaje final"
        ]

        inicio = time.time()
        for i, msg in enumerate(mensajes):
            await ws.send(msg)
            eco = await ws.recv()
            t = (time.time() - inicio) * 1000
            print(f"  [{t:6.1f} ms] Mensaje {i+1}: enviado y recibido")

        total = (time.time() - inicio) * 1000
        print(f"\n  Total para {len(mensajes)} mensajes: {total:.1f} ms")
        print(f"  Promedio por mensaje: {total/len(mensajes):.1f} ms")
        print(f"  Conexión reutilizada: Sí (una sola conexión TCP)")

await multiples_mensajes()

In [ ]:
# Simular una conversación (patrón común en chat apps)
async def conversacion():
    async with websockets.connect(ECHO_URL) as ws:
        print("Simulación de conversación via WebSocket:")
        print("-" * 40)

        dialogo = [
            "Hola, necesito ayuda",
            "Tengo un error en mi código",
            "Es un problema con la API",
        ]

        for msg in dialogo:
            await ws.send(msg)
            respuesta = await ws.recv()
            print(f"  Cliente -> {msg}")
            print(f"  Server  <- {respuesta}")
            print()

await conversacion()

## 5. Comparación con REST

Comparamos el rendimiento de enviar múltiples mensajes usando:
- **REST**: una petición HTTP por mensaje (nueva conexión cada vez)
- **WebSocket**: todos los mensajes por la misma conexión persistente

In [ ]:
N_MENSAJES = 10

# --- RTTs con REST (HTTP) ---
rtts_rest = []
for i in range(N_MENSAJES):
    inicio = time.time()
    respuesta = requests.post(
        "https://httpbin.org/post",
        json={"mensaje": f"ping-{i}"},
        timeout=10
    )
    rtt = (time.time() - inicio) * 1000
    rtts_rest.append(rtt)

print(f"REST ({N_MENSAJES} peticiones):")
print(f"  Promedio: {sum(rtts_rest)/len(rtts_rest):.1f} ms")
print(f"  Total:    {sum(rtts_rest):.1f} ms")

In [ ]:
# --- RTTs con WebSocket ---
rtts_ws = await medir_rtts(N_MENSAJES)

print(f"WebSocket ({N_MENSAJES} mensajes):")
print(f"  Promedio: {sum(rtts_ws)/len(rtts_ws):.1f} ms")
print(f"  Total:    {sum(rtts_ws):.1f} ms")

print(f"\nWebSocket es {sum(rtts_rest)/sum(rtts_ws):.1f}x más rápido en total")

## 6. Visualización

Comparamos visualmente los tiempos de ida y vuelta (RTT) de ambos protocolos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfica 1: RTT por mensaje ---
ax1 = axes[0]
x = range(1, N_MENSAJES + 1)
ax1.plot(x, rtts_rest, "o-", label="REST (HTTP)", color="#e74c3c", linewidth=2)
ax1.plot(x, rtts_ws, "s-", label="WebSocket", color="#2ecc71", linewidth=2)
ax1.set_xlabel("Número de mensaje")
ax1.set_ylabel("RTT (ms)")
ax1.set_title("RTT por mensaje")
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Gráfica 2: Tiempo acumulado ---
ax2 = axes[1]
rest_acumulado = [sum(rtts_rest[:i+1]) for i in range(len(rtts_rest))]
ws_acumulado = [sum(rtts_ws[:i+1]) for i in range(len(rtts_ws))]
ax2.plot(x, rest_acumulado, "o-", label="REST (HTTP)", color="#e74c3c", linewidth=2)
ax2.plot(x, ws_acumulado, "s-", label="WebSocket", color="#2ecc71", linewidth=2)
ax2.set_xlabel("Número de mensaje")
ax2.set_ylabel("Tiempo acumulado (ms)")
ax2.set_title("Tiempo acumulado")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("Comparación REST vs WebSocket", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Resumen final en barras
fig, ax = plt.subplots(figsize=(8, 5))

promedios = [
    sum(rtts_rest) / len(rtts_rest),
    sum(rtts_ws) / len(rtts_ws)
]
etiquetas = ["REST (HTTP)", "WebSocket"]
colores = ["#e74c3c", "#2ecc71"]

barras = ax.bar(etiquetas, promedios, color=colores, width=0.5)

for barra in barras:
    ax.text(
        barra.get_x() + barra.get_width() / 2.,
        barra.get_height() + 1,
        f"{barra.get_height():.1f} ms",
        ha="center", va="bottom", fontsize=12, fontweight="bold"
    )

ax.set_ylabel("RTT promedio (ms)")
ax.set_title(f"RTT promedio: REST vs WebSocket ({N_MENSAJES} mensajes)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()